# Stage 4 Baseline (Single Notebook)

This notebook contains the whole stage 4 pipeline in one place:
- Google Drive mount
- dependency installation
- dataset copy from Drive to local Colab runtime
- model training
- plot generation
- metric summary

Run it from top to bottom.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!python -m pip install -q numpy pandas Pillow matplotlib seaborn scikit-learn torch torchvision tqdm


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/MyProject1')
MANIFEST_DIR = PROJECT_ROOT / 'stage3_outputs_colab'
DRIVE_DATASET_ROOT = PROJECT_ROOT / 'DB'
LOCAL_RUNTIME_ROOT = Path('/content/MyProject1_runtime')
LOCAL_DATASET_ROOT = LOCAL_RUNTIME_ROOT / 'DB'
OUTPUT_DIR = PROJECT_ROOT / 'stage4_outputs'
MODEL_DIR = OUTPUT_DIR / 'models'
PLOT_DIR = OUTPUT_DIR / 'plots'
REPORT_DIR = OUTPUT_DIR / 'reports'

IMAGE_SIZE = 160
BATCH_SIZE = 256
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
RANDOM_SEED = 42
MODEL_NAME = 'resnet18'
USE_PRETRAINED = True
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

for directory in [OUTPUT_DIR, MODEL_DIR, PLOT_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT      =', PROJECT_ROOT)
print('MANIFEST_DIR      =', MANIFEST_DIR)
print('DRIVE_DATASET_ROOT=', DRIVE_DATASET_ROOT)
print('LOCAL_DATASET_ROOT=', LOCAL_DATASET_ROOT)
print('OUTPUT_DIR        =', OUTPUT_DIR)
print('DEVICE            =', DEVICE)


In [ ]:
assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'
assert DRIVE_DATASET_ROOT.exists(), f'Dataset folder not found: {DRIVE_DATASET_ROOT}'
assert MANIFEST_DIR.exists(), f'Manifest folder not found: {MANIFEST_DIR}'
assert (MANIFEST_DIR / 'train_manifest.csv').exists(), 'train_manifest.csv is missing'
assert (MANIFEST_DIR / 'val_manifest.csv').exists(), 'val_manifest.csv is missing'
assert (MANIFEST_DIR / 'test_manifest.csv').exists(), 'test_manifest.csv is missing'

print('Google Drive data is ready.')


In [ ]:
import shutil

%cd /content
LOCAL_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
if LOCAL_DATASET_ROOT.exists():
    shutil.rmtree(LOCAL_DATASET_ROOT)
shutil.copytree(DRIVE_DATASET_ROOT, LOCAL_DATASET_ROOT)
print('Dataset copied to local runtime:', LOCAL_DATASET_ROOT)


In [ ]:
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm


In [ ]:
class SteelDefectDataset:
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row['resolved_image_path']).convert('RGB')
        target = int(row['target'])
        if self.transform is not None:
            image = self.transform(image)
        return image, target


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def rebuild_image_path(image_path, source_folder, dataset_root):
    image_name = Path(image_path).name
    return str(dataset_root / 'images' / 'images' / source_folder / image_name)


def resolve_image_path(image_path, source_folder, dataset_root):
    raw_path = Path(image_path)
    if raw_path.exists():
        return str(raw_path)
    return rebuild_image_path(image_path, source_folder, dataset_root)


def load_manifest(csv_path, dataset_root):
    dataframe = pd.read_csv(csv_path)
    dataframe['resolved_image_path'] = dataframe.apply(
        lambda row: resolve_image_path(row['image_path'], row['source_folder'], dataset_root),
        axis=1,
    )
    return dataframe


def build_target_mapping(train_df):
    classes = (
        train_df[['class_id', 'class_name']]
        .drop_duplicates()
        .sort_values('class_id')
        .reset_index(drop=True)
    )
    class_id_to_target = {int(row.class_id): idx for idx, row in classes.iterrows()}
    target_to_class_name = {idx: row.class_name for idx, row in classes.iterrows()}
    target_to_class_id = {idx: int(row.class_id) for idx, row in classes.iterrows()}
    return class_id_to_target, target_to_class_name, target_to_class_id


def prepare_targets(dataframe, class_id_to_target):
    prepared = dataframe.copy()
    prepared['target'] = prepared['class_id'].map(class_id_to_target)
    return prepared


def validate_manifest_paths(dataframe, split_name):
    missing_paths = [path for path in dataframe['resolved_image_path'] if not Path(path).exists()]
    if missing_paths:
        preview = '\n'.join(f'  - {path}' for path in missing_paths[:10])
        raise FileNotFoundError(f'{split_name} has {len(missing_paths)} missing image files.\n{preview}')


def build_transforms(image_size):
    train_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    eval_transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_transform, eval_transform


def create_dataloaders(train_df, val_df, test_df, image_size, batch_size, num_workers, device):
    train_transform, eval_transform = build_transforms(image_size)
    pin_memory = str(device).startswith('cuda')
    train_dataset = SteelDefectDataset(train_df, transform=train_transform)
    val_dataset = SteelDefectDataset(val_df, transform=eval_transform)
    test_dataset = SteelDefectDataset(test_df, transform=eval_transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
    return train_loader, val_loader, test_loader


def get_model(model_name, num_classes, pretrained=True):
    if model_name != 'resnet18':
        raise ValueError('This notebook currently supports only resnet18.')
    weights = ResNet18_Weights.DEFAULT if pretrained else None
    model = resnet18(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def train_one_epoch(model, loader, criterion, optimizer, device, progress_bar=None):
    model.train()
    running_loss = 0.0
    all_targets = []
    all_preds = []
    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        preds = outputs.argmax(dim=1)
        running_loss += loss.item() * images.size(0)
        all_targets.extend(targets.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        if progress_bar is not None:
            progress_bar.update(1)
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    return epoch_loss, epoch_acc


def evaluate_model(model, loader, criterion, device, progress_bar=None):
    model.eval()
    running_loss = 0.0
    all_targets = []
    all_preds = []
    all_probs = []
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            preds = outputs.argmax(dim=1)
            probs = torch.softmax(outputs, dim=1).max(dim=1).values
            running_loss += loss.item() * images.size(0)
            all_targets.extend(targets.detach().cpu().numpy())
            all_preds.extend(preds.detach().cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())
            if progress_bar is not None:
                progress_bar.update(1)
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    return epoch_loss, epoch_acc, all_targets, all_preds, all_probs


def fit_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs, progress_bar=None):
    history_rows = []
    best_val_acc = -1.0
    best_state_dict = None
    for epoch in range(1, num_epochs + 1):
        epoch_started = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, progress_bar)
        val_loss, val_acc, _, _, _ = evaluate_model(model, val_loader, criterion, device, progress_bar)
        history_rows.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_accuracy': train_acc,
            'val_loss': val_loss,
            'val_accuracy': val_acc,
            'epoch_time_sec': time.time() - epoch_started,
        })
        print(f'Epoch {epoch:02d}/{num_epochs}: train_acc={train_acc:.4f}, val_acc={val_acc:.4f}')
        if progress_bar is not None:
            progress_bar.set_postfix(epoch=f'{epoch}/{num_epochs}', train_acc=f'{train_acc:.3f}', val_acc=f'{val_acc:.3f}')
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state_dict = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    history_df = pd.DataFrame(history_rows)
    model.load_state_dict(best_state_dict)
    return model, history_df, best_val_acc


def plot_history(history_df, output_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train_loss')
    axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val_loss')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[1].plot(history_df['epoch'], history_df['train_accuracy'], label='train_accuracy')
    axes[1].plot(history_df['epoch'], history_df['val_accuracy'], label='val_accuracy')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def plot_confusion_matrix(y_true, y_pred, class_names, output_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title('Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)


def save_predictions(test_df, y_true, y_pred, y_prob, target_to_class_name, output_path):
    predictions_df = test_df.copy().reset_index(drop=True)
    predictions_df['true_target'] = y_true
    predictions_df['pred_target'] = y_pred
    predictions_df['pred_class_name'] = [target_to_class_name[int(target)] for target in y_pred]
    predictions_df['pred_confidence'] = y_prob
    predictions_df.to_csv(output_path, index=False)


def build_test_metrics(y_true, y_pred, target_to_class_name, target_to_class_id, best_val_acc):
    labels = sorted(target_to_class_name.keys())
    class_names = [target_to_class_name[label] for label in labels]
    report = classification_report(y_true, y_pred, labels=labels, target_names=class_names, output_dict=True, zero_division=0)
    per_class = {}
    for label in labels:
        class_name = target_to_class_name[label]
        per_class[class_name] = {
            'target': int(label),
            'class_id': int(target_to_class_id[label]),
            'precision': float(report[class_name]['precision']),
            'recall': float(report[class_name]['recall']),
            'f1_score': float(report[class_name]['f1-score']),
            'support': int(report[class_name]['support']),
        }
    return {
        'best_val_accuracy': float(best_val_acc),
        'test_accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_avg_precision': float(report['macro avg']['precision']),
        'macro_avg_recall': float(report['macro avg']['recall']),
        'macro_avg_f1': float(report['macro avg']['f1-score']),
        'weighted_avg_precision': float(report['weighted avg']['precision']),
        'weighted_avg_recall': float(report['weighted avg']['recall']),
        'weighted_avg_f1': float(report['weighted avg']['f1-score']),
        'per_class': per_class,
    }


def save_json(payload, output_path):
    with open(output_path, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)


def save_report_template(output_path, test_accuracy, image_size, batch_size, num_epochs):
    text = f'''???? 4. ???????? ??????

??????: ResNet18
?????? ???????????: {image_size}
Batch size: {batch_size}
????: {num_epochs}
???????? test accuracy: {test_accuracy:.4f}
'''
    Path(output_path).write_text(text, encoding='utf-8')


def run_training_pipeline(project_root, dataset_root, manifest_dir, output_dir, image_size, batch_size, num_workers, num_epochs, learning_rate, weight_decay, random_seed, model_name, use_pretrained, device):
    set_seed(random_seed)
    train_csv = manifest_dir / 'train_manifest.csv'
    val_csv = manifest_dir / 'val_manifest.csv'
    test_csv = manifest_dir / 'test_manifest.csv'

    model_dir = output_dir / 'models'
    plot_dir = output_dir / 'plots'
    report_dir = output_dir / 'reports'
    for directory in [output_dir, model_dir, plot_dir, report_dir]:
        directory.mkdir(parents=True, exist_ok=True)

    train_df = load_manifest(train_csv, dataset_root)
    val_df = load_manifest(val_csv, dataset_root)
    test_df = load_manifest(test_csv, dataset_root)

    validate_manifest_paths(train_df, 'train')
    validate_manifest_paths(val_df, 'val')
    validate_manifest_paths(test_df, 'test')

    class_id_to_target, target_to_class_name, target_to_class_id = build_target_mapping(train_df)
    train_df = prepare_targets(train_df, class_id_to_target)
    val_df = prepare_targets(val_df, class_id_to_target)
    test_df = prepare_targets(test_df, class_id_to_target)

    train_loader, val_loader, test_loader = create_dataloaders(train_df, val_df, test_df, image_size, batch_size, num_workers, device)
    model = get_model(model_name=model_name, num_classes=len(class_id_to_target), pretrained=use_pretrained).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    total_steps = num_epochs * (len(train_loader) + len(val_loader)) + len(test_loader)
    with tqdm(total=total_steps, desc='Training progress', unit='batch') as progress_bar:
        model, history_df, best_val_acc = fit_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs, progress_bar)
        test_loss, test_acc, y_true, y_pred, y_prob = evaluate_model(model, test_loader, criterion, device, progress_bar)

    metrics = build_test_metrics(y_true, y_pred, target_to_class_name, target_to_class_id, best_val_acc)
    metrics['test_loss'] = float(test_loss)
    metrics['test_accuracy'] = float(test_acc)
    metrics['device'] = device
    metrics['model_name'] = model_name

    model_path = model_dir / f'{model_name}_best_model.pth'
    history_path = report_dir / 'history.csv'
    metrics_path = report_dir / 'test_metrics.json'
    class_mapping_path = report_dir / 'class_mapping.json'
    predictions_path = report_dir / 'test_predictions.csv'
    summary_path = report_dir / 'run_summary.json'
    report_template_path = report_dir / 'stage4_report_draft.txt'
    history_plot_path = plot_dir / 'training_history.png'
    confusion_matrix_path = plot_dir / 'confusion_matrix.png'

    torch.save(model.state_dict(), model_path)
    history_df.to_csv(history_path, index=False)
    save_json(metrics, metrics_path)
    save_json({str(target): {'class_id': int(target_to_class_id[target]), 'class_name': target_to_class_name[target]} for target in sorted(target_to_class_name.keys())}, class_mapping_path)
    save_predictions(test_df, y_true, y_pred, y_prob, target_to_class_name, predictions_path)
    plot_history(history_df, history_plot_path)
    plot_confusion_matrix(y_true, y_pred, [target_to_class_name[idx] for idx in sorted(target_to_class_name.keys())], confusion_matrix_path)
    save_report_template(report_template_path, metrics['test_accuracy'], image_size, batch_size, num_epochs)

    summary = {
        'project_root': str(project_root),
        'dataset_root': str(dataset_root),
        'manifest_dir': str(manifest_dir),
        'output_dir': str(output_dir),
        'device': device,
        'model_name': model_name,
        'use_pretrained': use_pretrained,
        'image_size': image_size,
        'batch_size': batch_size,
        'num_epochs': num_epochs,
        'learning_rate': learning_rate,
        'weight_decay': weight_decay,
        'num_workers': num_workers,
        'random_seed': random_seed,
        'trainable_parameters': count_parameters(model),
        'metrics': metrics,
        'artifacts': {
            'model_path': str(model_path),
            'history_path': str(history_path),
            'metrics_path': str(metrics_path),
            'class_mapping_path': str(class_mapping_path),
            'predictions_path': str(predictions_path),
            'history_plot_path': str(history_plot_path),
            'confusion_matrix_path': str(confusion_matrix_path),
        },
    }
    save_json(summary, summary_path)
    return summary, history_df


In [ ]:
summary, history_df = run_training_pipeline(
    project_root=PROJECT_ROOT,
    dataset_root=LOCAL_DATASET_ROOT,
    manifest_dir=MANIFEST_DIR,
    output_dir=OUTPUT_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    random_seed=RANDOM_SEED,
    model_name=MODEL_NAME,
    use_pretrained=USE_PRETRAINED,
    device=DEVICE,
)

print('\nTraining complete.')
print(f"Test accuracy: {summary['metrics']['test_accuracy']:.4f}")
print(f"Macro F1     : {summary['metrics']['macro_avg_f1']:.4f}")
print(f"Artifacts dir: {summary['output_dir']}")


In [ ]:
import json
from IPython.display import Image as IPyImage, display

summary_path = PROJECT_ROOT / 'stage4_outputs' / 'reports' / 'run_summary.json'
metrics_path = PROJECT_ROOT / 'stage4_outputs' / 'reports' / 'test_metrics.json'
history_plot_path = PROJECT_ROOT / 'stage4_outputs' / 'plots' / 'training_history.png'
confusion_matrix_path = PROJECT_ROOT / 'stage4_outputs' / 'plots' / 'confusion_matrix.png'

required_paths = [summary_path, metrics_path, history_plot_path, confusion_matrix_path]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    missing_preview = '\n'.join(str(path) for path in missing_paths)
    raise FileNotFoundError(
        'Training artifacts are not ready yet. Run the training cell successfully first.\n'
        f'Missing files:\n{missing_preview}'
    )

with open(summary_path, 'r', encoding='utf-8') as fp:
    summary = json.load(fp)
with open(metrics_path, 'r', encoding='utf-8') as fp:
    metrics = json.load(fp)

print('Device        :', summary['device'])
print('Model         :', summary['model_name'])
print('Test accuracy :', round(metrics['test_accuracy'], 4))
print('Macro F1      :', round(metrics['macro_avg_f1'], 4))
print('Output dir    :', summary['output_dir'])

display(IPyImage(filename=str(history_plot_path)))
display(IPyImage(filename=str(confusion_matrix_path)))
